# LangChain 기반 통신 요금제 추천 챗봇

기존 `chatbot_RAG` 프로젝트에서는 벡터 DB까지 사용했다. 이번 과제에서는 검색 결과가 만들어지는 과정을 직접 확인해 보기 위해 데이터와 검색 방식을 작게 줄였다.

질문과 관련 있는 요금제만 찾은 뒤 LangChain으로 답변을 만드는 것이 목표다. 요금제 정보는 학습용으로 정리한 샘플이므로 실제 가입 정보로 사용하지 않는다.

## 목차

1. 과제 목표와 처리 흐름
2. 환경 설정
3. 요금제 지식 데이터
4. 유사도 검색과 cutoff
5. 검색 결과 시각화
6. LangChain 프롬프트와 답변 체인
7. 테스트 및 결론
8. 사용자 정보 구조화 추천
9. 개선 및 빌드업 방향

## 1. 문제를 어떻게 나눴는가

처음에는 질문을 바로 모델에 전달할 수도 있지만, 그렇게 하면 모델이 샘플에 없는 요금제를 답할 수 있다. 그래서 검색과 답변 생성을 두 단계로 나눴다.

`질문 → TF-IDF 변환 → cosine similarity 계산 → cutoff 판정 → LangChain 답변`

- `search_plans()`는 관련 문서를 고른다. 이 단계의 결과는 같은 질문이면 항상 같다.
- `ask()`는 검색 결과가 있을 때만 모델을 호출한다. 검색 결과가 없으면 정해 둔 안내문을 반환한다.
- 답변 체인은 10주차 실습의 `prompt | model | parser` 형태를 그대로 사용했다.

이번 구현에서는 Agent, LangGraph, 대화 메모리, 도구 호출, 벡터 스토어를 넣지 않았다. 현재 문제를 푸는 데 필요하지 않고 검색 점수의 변화를 확인하기도 어려워지기 때문이다.

In [1]:
%pip -q install langchain langchain-openai scikit-learn pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 15.6 MB/s eta 0:00:0000:01


## 2. 환경 설정

10주차 `Langchain_Example.ipynb`에서 사용한 방식에 맞춰 모델은 `init_chat_model()`로 생성한다. API 키는 코드에 적지 않고 Colab **Secrets**의 `OPENAI_API_KEY`에서 읽는다. 검색과 그래프는 API 키 없이 동작하고, 실제 답변을 만들 때만 모델 호출이 발생한다.

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import userdata
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
plt.rcParams["axes.unicode_minus"] = False

TimeoutException: Requesting secret OPENAI_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

## 3. 요금제 데이터를 어떻게 구성했는가

제출 파일만 열어도 실행할 수 있도록 대표 요금제 9개를 리스트에 직접 넣었다. 외부 JSON을 읽는 코드까지 추가하면 파일 경로가 달라졌을 때 실행이 끊기기 때문이다.

처음에는 요금제 이름만 비교하는 방법을 생각했지만, 사용자는 보통 가격이나 데이터량, OTT 혜택으로 질문한다. 그래서 각 딕셔너리의 모든 값을 하나의 `검색문서` 문자열로 합쳤다. 이 구조에서는 `넷플릭스`, `청소년`, `테더링` 같은 조건도 유사도 계산에 반영된다.

In [ ]:
plans = [
    {"통신사": "SKT", "요금제": "0 청년 59", "가격": "59,000원", "데이터": "36GB, 소진 후 속도제어", "혜택": "Wavve 할인, 커피·영화·로밍 50% 할인", "추천대상": "만 34세 이하 청년"},
    {"통신사": "SKT", "요금제": "0 청년 109(넷플릭스)", "가격": "109,000원", "데이터": "무제한", "혜택": "넷플릭스 스탠다드 무료", "추천대상": "넷플릭스를 자주 보는 청년"},
    {"통신사": "SKT", "요금제": "다이렉트5G 48", "가격": "48,000원", "데이터": "110GB, 소진 후 속도제어", "혜택": "온라인 전용, Wavve 할인", "추천대상": "약정 없이 데이터를 많이 쓰는 사용자"},
    {"통신사": "KT", "요금제": "Y 슬림", "가격": "55,000원", "데이터": "14GB, 소진 후 속도제어", "혜택": "음성 무제한", "추천대상": "만 34세 이하, 월 10~15GB 사용자"},
    {"통신사": "KT", "요금제": "5G 슬림 10GB", "가격": "50,000원", "데이터": "10GB, 소진 후 속도제어", "혜택": "음성 무제한", "추천대상": "와이파이 중심의 가벼운 데이터 사용자"},
    {"통신사": "KT", "요금제": "초이스 베이직(넷플릭스)", "가격": "90,000원", "데이터": "무제한", "혜택": "넷플릭스 베이식 포함", "추천대상": "데이터와 넷플릭스를 함께 쓰는 사용자"},
    {"통신사": "LGU+", "요금제": "5G 라이트 청소년", "가격": "45,000원", "데이터": "8GB, 소진 후 1Mbps", "혜택": "U+모바일tv 라이트", "추천대상": "와이파이를 주로 쓰는 청소년"},
    {"통신사": "LGU+", "요금제": "5G 데이터 레귤러", "가격": "63,000원", "데이터": "50GB, 소진 후 1Mbps", "혜택": "디즈니+ 스탠다드 40% 할인", "추천대상": "데이터를 꾸준히 쓰는 실속형 사용자"},
    {"통신사": "LGU+", "요금제": "5G 데이터 슈퍼", "가격": "68,000원", "데이터": "95GB, 소진 후 3Mbps", "혜택": "디즈니+ 스탠다드 40% 할인", "추천대상": "스트리밍과 테더링을 즐기는 사용자"},
]

documents = []
for plan in plans:
    documents.append(" | ".join(f"{key}: {value}" for key, value in plan.items()))

df = pd.DataFrame(plans)
df["검색문서"] = documents
df.drop(columns="검색문서")

## 4. 유사도 검색에서 사용한 기준

### TF-IDF

TF-IDF는 한 문서에 자주 나오면서 다른 문서에는 흔하지 않은 표현에 더 큰 값을 준다. 한국어는 조사와 띄어쓰기에 따라 단어가 달라질 수 있어, 단어 대신 문자 2~4개 묶음인 `char_wb` n-gram을 사용했다.

### Cosine similarity

질문 벡터와 문서 벡터가 같은 방향에 가까울수록 점수가 높아진다. 문장 길이 자체보다 포함된 표현의 방향을 비교할 수 있어 길이가 다른 질문과 요금제 설명을 비교하기에 적합했다.

### cutoff와 Top K

계산한 점수가 `0.15` 이상인 문서만 남기고 그중 상위 2개를 선택했다. Top K만 적용하면 전혀 관련 없는 질문에도 결과가 생긴다. cutoff를 함께 둔 이유가 여기에 있다. `0.15`는 현재 샘플 질문에서 관련 질문은 남고 `강아지 사료` 질문은 제외되는지 확인해 정한 실습용 값이다.

In [ ]:
CUTOFF = 0.15
TOP_K = 2

# ponytail: 9개 샘플은 메모리 검색으로 충분하며, 수백 개 이상이면 임베딩/벡터 DB를 검토한다.
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
document_vectors = vectorizer.fit_transform(df["검색문서"])

def search_plans(question, cutoff=CUTOFF, top_k=TOP_K):
    scores = cosine_similarity(vectorizer.transform([question]), document_vectors)[0]
    result = df.assign(유사도=scores)
    result = result[result["유사도"] >= cutoff]
    return result.sort_values("유사도", ascending=False).head(top_k)

question = "넷플릭스가 포함된 데이터 무제한 요금제 추천해줘"
result = search_plans(question)
result[["통신사", "요금제", "가격", "유사도"]]

## 5. 점수를 그래프로 확인한 이유

검색 결과 표만 보면 선택된 두 문서만 확인할 수 있다. 모든 문서의 점수를 막대그래프로 그리고 cutoff를 빨간 점선으로 표시해 탈락한 문서와 점수 차이도 확인했다.

여기서 확인할 부분은 점수의 절댓값보다 후보 사이의 차이다. 선택 문서와 탈락 문서의 점수가 계속 비슷하다면 cutoff나 검색 문서를 다시 조정해야 한다.

In [ ]:
def visualize_scores(question, cutoff=CUTOFF):
    scores = cosine_similarity(vectorizer.transform([question]), document_vectors)[0]
    labels = [f"{carrier} P{i + 1}" for i, carrier in enumerate(df["통신사"])]
    chart = pd.DataFrame({"label": labels, "score": scores})
    chart = chart.sort_values("score")
    colors = ["#2563eb" if score >= cutoff else "#cbd5e1" for score in chart["score"]]
    ax = chart.plot.barh(x="label", y="score", color=colors, legend=False, figsize=(9, 5))
    ax.axvline(cutoff, color="#dc2626", linestyle="--", label=f"cutoff = {cutoff}")
    ax.set(title="Plan similarity scores", xlabel="cosine similarity", ylabel="")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print("질문:", question)

visualize_scores(question)

## 6. 프롬프트와 체인을 이렇게 작성한 이유

검색 함수가 후보를 고르더라도 모델이 자기 지식을 섞으면 앞 단계의 의미가 없어진다. 따라서 시스템 메시지에 다음 규칙을 넣었다.

- 전달받은 검색 결과만 근거로 사용한다.
- 가격, 데이터, 혜택을 같이 비교한다.
- 검색 결과에 없는 내용은 추측하지 않는다.
- 추천 개수는 최대 2개로 제한한다.

`temperature=0`은 같은 자료로 답변할 때 표현의 변동을 줄이기 위한 선택이다. 마지막의 `StrOutputParser`는 모델 응답에서 문자열 내용만 꺼내므로 출력 코드를 짧게 유지할 수 있다.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 통신 요금제 추천 상담사입니다.
반드시 제공된 검색 결과만 근거로 답하세요.
후보의 가격, 데이터, 혜택을 비교해 최대 2개만 추천하세요.
검색 결과에 없는 내용은 추측하지 말고 '제공된 정보에서 확인할 수 없습니다'라고 답하세요.
마지막에 '실제 가입 전 공식 정보를 확인하세요'라고 안내하세요."""),
    ("human", "질문: {question}\n\n검색 결과:\n{context}"),
])

model = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)
chain = prompt | model | StrOutputParser()

def ask(question, cutoff=CUTOFF):
    matches = search_plans(question, cutoff=cutoff)
    if matches.empty:
        return "관련 요금제를 찾지 못했습니다. 질문에 통신사, 데이터량 또는 원하는 혜택을 추가해 주세요."
    context = "\n".join(matches["검색문서"])
    return chain.invoke({"question": question, "context": context})

ask(question)

## 7. 검색 로직을 확인한 방법

검색 함수에는 cutoff 분기가 있으므로 두 가지 경우를 확인했다. `청소년 5G 요금제`는 결과가 있어야 하고, 데이터와 무관한 `강아지 사료` 질문은 빈 결과가 되어야 한다. 이 두 `assert`가 통과하면 최소한 관련 질문 통과와 무관 질문 차단 흐름은 유지되고 있다고 볼 수 있다.

아래 세 질문은 대상, 사용 패턴, 무관 질문으로 조건을 다르게 잡았다. 출력 내용을 보면서 검색된 요금제의 가격·데이터·혜택이 답변에 그대로 사용됐는지 확인한다.

In [ ]:
assert not search_plans("청소년 5G 요금제").empty
assert search_plans("강아지 사료 추천해줘").empty
print("cutoff 검색 테스트 통과")

for sample in [
    "청소년이 쓸 저렴한 5G 요금제 알려줘",
    "테더링과 스트리밍에 맞는 데이터 많은 요금제 추천해줘",
    "강아지 사료 추천해줘",
]:
    print(f"\nQ. {sample}\nA. {ask(sample)}")

### 구현하면서 확인한 점

- 데이터가 9개뿐인 경우에는 별도 벡터 DB 없이 메모리에서 유사도를 계산해도 흐름을 확인하기에 충분했다.
- Top K는 항상 결과를 만들지만 cutoff는 결과를 반환하지 않을 선택을 가능하게 한다. 두 값의 역할이 달랐다.
- 검색은 후보를 고르는 역할이고 LangChain은 그 후보를 읽기 쉬운 답변으로 정리하는 역할이었다. 두 단계를 나누니 오류가 검색 문제인지 답변 문제인지 구분하기 쉬웠다.
- 프롬프트를 자세히 작성해도 잘못된 문서가 검색되면 답변도 좋아지기 어렵다. 모델 설정 전에 검색 결과를 먼저 확인해야 했다.

## 8. 사용자 정보 구조화 추천

자유 문장도 검색은 되지만, 사용자가 나이·선호 OTT·데이터 습관을 한 문장에 섞어 말하면 문서에 쓰인 표현(`청년`, `청소년` 같은 범주어)과 어긋나 점수가 낮게 나올 수 있다. `model.with_structured_output()`으로 이런 표현을 먼저 뽑아 검색어에 더해주면 `search_plans()`/`ask()`는 그대로 재사용할 수 있다.

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field

class UserProfile(BaseModel):
    연령대: Optional[str] = Field(None, description="요금제 문서에 쓰이는 연령대 표현(예: 청소년, 청년)")
    선호_OTT: Optional[str] = Field(None, description="선호하는 OTT 서비스명")
    데이터사용패턴: Optional[str] = Field(None, description="데이터 사용 습관을 요약한 짧은 문장")

profile_extractor = model.with_structured_output(UserProfile)

def recommend_for_user(user_info):
    profile = profile_extractor.invoke(
        f"다음 사용자 설명에서 요금제 추천에 필요한 정보만 뽑아줘. 없으면 null로 둬.\n설명: {user_info}"
    )
    query = " ".join(filter(None, [profile.연령대, profile.선호_OTT, profile.데이터사용패턴, user_info]))
    return profile, ask(query)

In [ ]:
# ponytail: 검색어 조합 자체는 LLM 호출 없이도 확인할 수 있다.
sample_profile = UserProfile(연령대="청년", 선호_OTT="넷플릭스")
sample_query = " ".join(filter(None, [sample_profile.연령대, sample_profile.선호_OTT, sample_profile.데이터사용패턴, "30살인데 넷플릭스를 챙겨봐요"]))
assert not search_plans(sample_query).empty
print("구조화 검색어 테스트 통과")

In [ ]:
for user_info in [
    "30살 청년인데 넷플릭스를 챙겨봐요",
    "청소년인데 와이파이 위주로 가볍게 씁니다",
]:
    profile, answer = recommend_for_user(user_info)
    print(f"\n사용자: {user_info}\n추출된 정보: {profile}\n답변: {answer}")

## 9. 다음에 고친다면

현재 코드에서 먼저 보완할 부분은 벡터 DB가 아니라 검색 기준이다.

1. 대표 질문을 더 모아 질문별로 기대하는 요금제를 적는다.
2. cutoff를 바꿔 보면서 기대한 요금제가 남는지, 무관 질문이 차단되는지 비교한다.
3. `6만원 이하`, `80GB 이상`처럼 숫자 비교가 필요한 질문이 많아지면 가격과 데이터량을 숫자로 분리해 먼저 필터링한다.
4. 같은 뜻의 다른 표현을 놓치는 사례가 쌓이면 TF-IDF를 임베딩 검색으로 교체한다.
5. 문서가 수백 개 이상으로 늘고 저장·갱신이 필요해질 때 벡터 DB를 검토한다.

`ask()`는 `search_plans()`의 결과만 받으므로 답변 체인을 바꾸지 않고 검색 함수부터 개선할 수 있다. 지금 데이터 크기에서는 현재 방식으로 실패 질문을 모으는 것이 다음 기능을 미리 추가하는 것보다 우선이라고 판단했다.